# 1. Libraries


In [32]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns

sns.set_style("whitegrid")

# 2. Data Path and Loading

In [33]:
DATA_DIR = Path("../data/eye_tracking")

In [34]:
participant_id = "002"

In [35]:
dfs = []

for condition in [1, 2, 3]:
    matches = list(DATA_DIR.glob(f"{participant_id}_ET_Data_Condition{condition}_*.csv"))

    if not matches:
        raise FileNotFoundError(
            f"No file found for participant {participant_id}, condition {condition}"
        )

    file_path = matches[0]

    df_condition = pd.read_csv(file_path, low_memory=False)
    df_condition["participant_id"] = participant_id
    df_condition["condition_number"] = condition

    dfs.append(df_condition)

df = pd.concat(dfs, ignore_index=True)
df.head()


,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,trial_number,model_visibility_state,object_position_x,object_position_y,object_position_z,object_rotation_x,object_rotation_y,object_rotation_z,object_rotation_w,participant_id
0,1000002129744661600,1778070220159,509.0569,2.0,285574,0.0,Valid,-0.028935,0.091456,0.995389,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,002
1,1000002129749661400,1778070220160,509.0569,2.0,285575,0.0,Valid,-0.031357,0.088013,0.995626,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,002
2,1000002129754661500,1778070220160,509.0569,2.0,285576,0.0,Valid,-0.033649,0.085436,0.995775,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,002
3,1000002129759663900,1778070220160,509.0569,2.0,285577,0.0,Valid,-0.035741,0.083344,0.995880,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,002
4,1000002129764664000,1778070220160,509.0569,2.0,285578,0.0,Valid,-0.037969,0.081460,0.995953,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,002


In [36]:
def add_time_per_trial(df, time_col="gaze_capture_time"):
    df = df.copy()

    df = df.sort_values(["condition_number", "trial_number", time_col])

    trial_start = df.groupby(["condition_number", "trial_number"])[time_col].transform("min")

    df["time_ms"] = (df[time_col] - trial_start) / 1_000_000

    return df

In [37]:
df = add_time_per_trial(df, time_col="gaze_capture_time")

In [18]:
df

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,model_visibility_state,object_position_x,object_position_y,object_position_z,object_rotation_x,object_rotation_y,object_rotation_z,object_rotation_w,participant_id,time_ms
0,1000001481958514100,1778060264384,592.3813,2.000000,232023,0.000000,Valid,-0.097741,0.230809,0.968077,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,0.0000
1,1000001481963514900,1778060264384,592.3813,2.000000,232024,0.000000,Valid,-0.097410,0.230441,0.968198,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,5.0008
2,1000001481968515700,1778060264384,592.3813,2.000000,232025,0.000000,Valid,-0.097012,0.229992,0.968345,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,10.0016
3,1000001481973516500,1778060264384,592.3813,2.000000,232026,0.000000,Valid,-0.096599,0.229737,0.968447,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,15.0024
4,1000001481978516800,1778060264384,592.3813,2.000000,232027,0.000000,Valid,-0.096269,0.230169,0.968377,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,20.0027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553473,1000005111509028100,1778063893701,3746.4670,1.157945,957652,0.027887,Valid,0.076445,0.145445,0.986409,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,68516.8066
553474,1000005111514028600,1778063893701,3746.4670,1.148934,957653,0.000000,Valid,0.075853,0.145398,0.986461,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,68521.8071
553475,1000005111519029800,1778063893711,3746.4780,1.150903,957654,0.000000,Valid,0.075171,0.145675,0.986472,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,68526.8083
553476,1000005111524031000,1778063893711,3746.4780,1.141437,957655,0.000000,Valid,0.074669,0.146021,0.986460,...,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001,68531.8095


In [38]:
FORWARD_COLS = ["gaze_forward_x", "gaze_forward_y", "gaze_forward_z"]
CLEAN_FORWARD_COLS = ["clean_gaze_forward_x", "clean_gaze_forward_y", "clean_gaze_forward_z"]


def good_status(left_status: pd.Series, right_status: pd.Series) -> np.ndarray:
    l = left_status.astype(str).str.lower().str.strip()
    r = right_status.astype(str).str.lower().str.strip()

    allowed_pairs = {
        ("tracked", "tracked"),
        ("tracked", "compensated"),
        ("compensated", "tracked"),
        ("compensated", "compensated")
    }

    good = pd.Series(
        [(ls, rs) in allowed_pairs for ls, rs in zip(l, r)],
        index=left_status.index
    )

    return good.to_numpy(dtype=bool)

def find_runs(mask: np.ndarray):
    mask = np.asarray(mask, dtype=bool)
    idx = np.flatnonzero(mask)

    if len(idx) == 0:
        return []

    runs = []
    start = prev = idx[0]

    for k in idx[1:]:
        if k == prev + 1:
            prev = k
        else:
            runs.append((start, prev))
            start = prev = k

    runs.append((start, prev))
    return runs

def expand_runs(runs, n, pad=2):
    """
    Expand each run by `pad` samples on both sides,
    then merge overlaps.
    """
    if len(runs) == 0:
        return []

    expanded = []
    for a, b in runs:
        aa = max(0, a - pad)
        bb = min(n - 1, b + pad)
        expanded.append((aa, bb))

    expanded.sort()
    merged = [expanded[0]]

    for a, b in expanded[1:]:
        prev_a, prev_b = merged[-1]
        if a <= prev_b + 1:
            merged[-1] = (prev_a, max(prev_b, b))
        else:
            merged.append((a, b))

    return merged


def normalize_xyz(xyz: np.ndarray) -> np.ndarray:
    xyz = np.asarray(xyz, dtype=float)
    nrm = np.linalg.norm(xyz, axis=1)
    out = np.full_like(xyz, np.nan, dtype=float)
    good = np.isfinite(nrm) & (nrm > 1e-8) & np.all(np.isfinite(xyz), axis=1)
    out[good] = xyz[good] / nrm[good, None]
    return out


def interpolate_segment(sub: pd.DataFrame, forward_cols=FORWARD_COLS, pad: int = 2) -> pd.DataFrame:
    sub = sub.sort_values("time_ms").copy()
    t = sub["time_ms"].to_numpy(dtype=float)

    xyz = np.column_stack([
        pd.to_numeric(sub[c], errors="coerce").to_numpy(dtype=float)
        for c in forward_cols
    ])

    good = good_status(sub["left_status"], sub["right_status"])
    bad = ~good
    bad_runs = find_runs(bad)

    n = len(sub)
    is_interpolated = np.zeros(n, dtype=bool)

    clean = xyz.copy()

    # expand bad runs by buffer
    buffered_runs = expand_runs(bad_runs, n=n, pad=pad)

    # set buffered regions to NaN before interpolation
    for a, b in buffered_runs:
        clean[a:b+1, :] = np.nan

    for a, b in buffered_runs:
        left = a - 1
        right = b + 1

        bracketed = (
            left >= 0 and
            right < n and
            np.all(np.isfinite(xyz[left])) and
            np.all(np.isfinite(xyz[right]))
        )

        if bracketed:
            for dim in range(3):
                clean[a:b+1, dim] = np.interp(
                    t[a:b+1],
                    [t[left], t[right]],
                    [xyz[left, dim], xyz[right, dim]]
                )

            clean[a:b+1, :] = normalize_xyz(clean[a:b+1, :])
            is_interpolated[a:b+1] = True

    sub["good_sample"] = good
    sub["bad_sample"] = bad
    sub["is_interpolated"] = is_interpolated

    for j, c in enumerate(forward_cols):
        sub[f"clean_{c}"] = clean[:, j]

    return sub


def interpolate_all_segments(df: pd.DataFrame, pad: int = 2) -> pd.DataFrame:
    out = []

    for (cond, trial), sub in df.groupby(["condition_number", "trial_number"], sort=True):
        cleaned = interpolate_segment(sub, forward_cols=FORWARD_COLS, pad=pad)
        cleaned["segment_label"] = f"C{int(cond)}_T{int(trial)}"
        out.append(cleaned)

    return (
        pd.concat(out, axis=0)
        .sort_values(["condition_number", "trial_number", "time_ms"])
        .reset_index(drop=True)
    )

In [39]:
df = interpolate_all_segments(df)

In [40]:
def median_filter_valid_runs(x: pd.Series, window: int = 5) -> pd.Series:
    y = x.to_numpy(dtype=float).copy()
    out = y.copy()

    valid = np.isfinite(y)
    idx = np.flatnonzero(valid)

    if len(idx) == 0:
        return pd.Series(out, index=x.index)

    runs = []
    start = prev = idx[0]

    for k in idx[1:]:
        if k == prev + 1:
            prev = k
        else:
            runs.append((start, prev))
            start = prev = k
    runs.append((start, prev))

    for a, b in runs:
        seg = pd.Series(y[a:b+1])

        # centered 5-point median, allowing shorter windows at edges
        filt = seg.rolling(window=window, center=True, min_periods=1).median()

        out[a:b+1] = filt.to_numpy()

    return pd.Series(out, index=x.index)

WINDOW = 5

for col in CLEAN_FORWARD_COLS:
    df[col] = (
        df.groupby(["condition_number", "trial_number"])[col]
        .transform(lambda x: median_filter_valid_runs(x, window=WINDOW))
    )

# renormalize after median filtering
xyz = df[CLEAN_FORWARD_COLS].to_numpy(dtype=float)
df[CLEAN_FORWARD_COLS] = normalize_xyz(xyz)

In [41]:
# 7) QUALITY-CONTROL SUMMARY
qc = (
    df.groupby(["condition_number", "trial_number"])
    .agg(
        n_samples=("time_ms", "size"),
        n_bad=("bad_sample", "sum"),
        n_interpolated=("is_interpolated", "sum"),
    )
    .reset_index()
)

# percentages
qc["pct_bad"] = 100 * qc["n_bad"] / qc["n_samples"]
qc["pct_interpolated"] = 100 * qc["n_interpolated"] / qc["n_samples"]

print(qc.to_string(index=False))

 condition_number  trial_number  n_samples  n_bad  n_interpolated  pct_bad  pct_interpolated
                1             0      27561    272             432 0.986902          1.567432
                1             1      51120   1583            2127 3.096635          4.160798
                1             2      23768    619             863 2.604342          3.630932
                1             3       9212    153             221 1.660877          2.399045
                1             4      15092    466             638 3.087729          4.227405
                1             5      20784    506             717 2.434565          3.449769
                1             6      25512    629             841 2.465506          3.296488
                2             0      27620    784            1039 2.838523          3.761767
                2             1      53692   1718            2199 3.199732          4.095582
                2             2      16224    351             465 2.16

In [42]:
def add_angular_velocity(df, time_col="gaze_capture_time"):
    """
    Compute gaze angular velocity (deg/s) from Varjo gaze_forward vectors.

    Method
    ------
    1. Take consecutive 3D gaze direction vectors.
    2. Compute the angle between them with:
           angle = atan2(||g_prev x g_curr||, g_prev · g_curr)
       This is numerically stable.
    3. Divide by the time difference in seconds.
    4. Convert from rad/s to deg/s.

    Assumptions
    -----------
    - gaze_forward_x/y/z describe gaze direction.
    - time_col is in nanoseconds.
    - Velocity should be computed within each participant/model block.

    Returns
    -------
    df : copy of input dataframe with a new column:
         - angular_velocity
    """
    df = df.copy()

    # Sort so consecutive rows are truly consecutive in time
    df = df.sort_values(
        ["condition_number", "trial_number", time_col]
    )

    # Initialize output column
    df["angular_velocity"] = np.nan

    # Compute separately within each continuous block
    for (_, _), sub in df.groupby(["condition_number", "trial_number"], sort=False):
        idx = sub.index

        # Extract gaze direction vectors
        g = sub[[
            "clean_gaze_forward_x",
            "clean_gaze_forward_y",
            "clean_gaze_forward_z"
        ]].to_numpy(dtype=float)

        # Normalize vectors just in case they are not perfectly unit length
        norms = np.linalg.norm(g, axis=1, keepdims=True)
        g = g / np.clip(norms, 1e-12, None)

        # Previous and current vectors
        g_prev = g[:-1]
        g_curr = g[1:]

        # Angle between consecutive vectors:
        # angle = atan2(||cross||, dot)
        cross_norm = np.linalg.norm(np.cross(g_prev, g_curr), axis=1)
        dot_prod = np.sum(g_prev * g_curr, axis=1)
        angles_rad = np.arctan2(cross_norm, dot_prod)

        # Time difference in seconds (timestamps are in nanoseconds)
        t = sub[time_col].to_numpy(dtype=np.int64)
        dt = np.diff(t) * 1e-9

        # Prepare output for this block
        vel = np.full(len(sub), np.nan)

        # Only compute where dt is valid
        valid = dt > 0
        vel[1:][valid] = np.degrees(angles_rad[valid] / dt[valid])

        # Write back into dataframe
        df.loc[idx, "angular_velocity"] = vel

    return df

In [43]:
df = add_angular_velocity(df, time_col="gaze_capture_time")

In [52]:
def at_mad_threshold(angular_vel, th_0=200.0, min_samples=10, tol=1.0, max_iter=50):
    """
    Compute one adaptive saccade threshold from angular velocity using MAD.

    Parameters
    ----------
    angular_vel : array-like
        Angular velocity values in deg/s.
    th_0 : float, default=200.0
        Initial threshold in deg/s.
    min_samples : int, default=10
        Minimum number of valid samples required.
    tol : float, default=1.0
        Convergence tolerance in deg/s.
    max_iter : int, default=50
        Maximum number of iterations.

    Returns
    -------
    saccade_thresh : float
        Final threshold in deg/s.
    threshs : list[float]
        Threshold values across iterations.
    """
    v = pd.to_numeric(pd.Series(angular_vel), errors="coerce").to_numpy(dtype=float)
    v = v[np.isfinite(v)]

    if len(v) < min_samples:
        return np.nan, []

    threshs = []
    current_th = float(th_0)

    for _ in range(max_iter):
        threshs.append(current_th)

        subset = v[v < current_th]
        subset = subset[np.isfinite(subset)]

        if len(subset) < min_samples:
            return np.nan, threshs

        median = np.median(subset)
        mad = np.median(np.abs(subset - median))
        next_th = median + 3.0 * 1.486 * mad

        if not np.isfinite(next_th):
            return np.nan, threshs

        if abs(current_th - next_th) <= tol:
            threshs.append(float(next_th))
            return float(next_th), threshs

        current_th = float(next_th)

    return float(current_th), threshs


def compute_global_mad_threshold(
    df,
    vel_col="angular_velocity",
    th_0=200.0,
    min_samples=10,
    tol=1.0,
    max_iter=50,
):
    """
    Compute one MAD-based threshold across all trials, conditions, and rows.

    Returns
    -------
    threshold : float
        Global angular velocity threshold.
    summary_df : pd.DataFrame
        One-row summary table for reporting/saving.
    """
    threshold, trace = at_mad_threshold(
        df[vel_col],
        th_0=th_0,
        min_samples=min_samples,
        tol=tol,
        max_iter=max_iter,
    )

    n_valid = pd.to_numeric(df[vel_col], errors="coerce").notna().sum()

    summary_df = pd.DataFrame(
        {
            "mad_threshold": [threshold],
            "n_valid_velocity_samples": [int(n_valid)],
            "threshold_trace": [trace],
        }
    )

    return threshold, summary_df


def add_global_mad_threshold(df, threshold, threshold_col="mad_threshold"):
    """
    Add the same global threshold to every row.
    """
    df = df.copy()
    df[threshold_col] = threshold
    return df


In [64]:
global_threshold, threshold_summary = compute_global_mad_threshold(df)

In [65]:
df = add_global_mad_threshold(df, global_threshold)

In [66]:
print("Global MAD threshold:", global_threshold)

Global MAD threshold: 44.96111887090785


In [67]:
threshold_summary

,mad_threshold,n_valid_velocity_samples,threshold_trace
0,44.961119,493807,"[200.0, 71.23667754232585, 54.71772205875763, ..."


In [68]:
def add_velocity_category(
    df,
    vel_col="angular_velocity",
    threshold_col="mad_threshold",
    category_col="velocity_category",
):
    """
    Label each sample as fixation or saccade using angular velocity and MAD threshold.

    Rules
    -----
    - fixation: angular_velocity < mad_threshold
    - saccade:  angular_velocity >= mad_threshold
    - NaN:      if angular_velocity or mad_threshold is missing
    """
    df = df.copy()

    vel = pd.to_numeric(df[vel_col], errors="coerce")
    thr = pd.to_numeric(df[threshold_col], errors="coerce")

    df[category_col] = pd.Series(pd.NA, index=df.index, dtype="string")

    valid = vel.notna() & thr.notna()
    df.loc[valid & (vel < thr), category_col] = "fixation"
    df.loc[valid & (vel >= thr), category_col] = "saccade"

    return df

In [69]:
df = add_velocity_category(df)

In [70]:
df

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,good_sample,bad_sample,is_interpolated,clean_gaze_forward_x,clean_gaze_forward_y,clean_gaze_forward_z,segment_label,angular_velocity,mad_threshold,velocity_category
0,1000002129744661600,1778070220159,509.0569,2.0,285574,0.000000,Valid,-0.028935,0.091456,0.995389,...,True,False,False,-0.031357,0.088013,0.995626,C1_T0,NaN,44.961119,<NA>
1,1000002129749661400,1778070220160,509.0569,2.0,285575,0.000000,Valid,-0.031357,0.088013,0.995626,...,True,False,False,-0.032503,0.086725,0.995702,C1_T0,19.783303,44.961119,fixation
2,1000002129754661500,1778070220160,509.0569,2.0,285576,0.000000,Valid,-0.033649,0.085436,0.995775,...,True,False,False,-0.033649,0.085436,0.995775,C1_T0,19.782116,44.961119,fixation
3,1000002129759663900,1778070220160,509.0569,2.0,285577,0.000000,Valid,-0.035741,0.083344,0.995880,...,True,False,False,-0.035741,0.083344,0.995880,C1_T0,33.904964,44.961119,fixation
4,1000002129764664000,1778070220160,509.0569,2.0,285578,0.000000,Valid,-0.037969,0.081460,0.995953,...,True,False,False,-0.037969,0.081460,0.995953,C1_T0,33.445487,44.961119,fixation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493848,1000005499218055600,1778073589410,3331.1760,2.0,959210,0.247830,Valid,-0.015834,0.193677,0.980938,...,True,False,False,-0.015835,0.193311,0.981010,C3_T6,22.325030,44.961119,fixation
493849,1000005499223058100,1778073589410,3331.1760,2.0,959211,0.227171,Valid,-0.013958,0.193113,0.981077,...,True,False,False,-0.013958,0.193113,0.981077,C3_T6,21.633492,44.961119,fixation
493850,1000005499228059100,1778073589421,3331.1870,2.0,959212,0.238938,Valid,-0.012489,0.192788,0.981161,...,True,False,False,-0.012489,0.192788,0.981161,C3_T6,17.259809,44.961119,fixation
493851,1000005499233060100,1778073589421,3331.1870,2.0,959213,0.245437,Valid,-0.011041,0.192361,0.981262,...,True,False,False,-0.011765,0.192574,0.981212,C3_T6,8.669213,44.961119,fixation


In [12]:
df_condition

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,trial_number,model_visibility_state,object_position_x,object_position_y,object_position_z,object_rotation_x,object_rotation_y,object_rotation_z,object_rotation_w,participant_id
0,1000004154588007200,1778062937116,2790.398,0.351046,766342,0.000000,Valid,0.298656,0.159875,0.940874,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
1,1000004154888038800,1778062937116,2790.398,0.452291,766402,0.000000,Valid,0.263659,0.158534,0.951500,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
2,1000004154893038900,1778062937116,2790.398,0.107576,766403,0.000000,Valid,0.248639,0.160037,0.955284,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
3,1000004154898038300,1778062937117,2790.398,0.189546,766404,0.000000,Valid,0.244448,0.160614,0.956268,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
4,1000004154903038800,1778062937117,2790.398,0.312154,766405,0.000000,Valid,0.230536,0.161675,0.959538,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184438,1000005111509028100,1778063893701,3746.467,1.157945,957652,0.027887,Valid,0.076445,0.145445,0.986409,...,6,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
184439,1000005111514028600,1778063893701,3746.467,1.148934,957653,0.000000,Valid,0.075853,0.145398,0.986461,...,6,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
184440,1000005111519029800,1778063893711,3746.478,1.150903,957654,0.000000,Valid,0.075171,0.145675,0.986472,...,6,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
184441,1000005111524031000,1778063893711,3746.478,1.141437,957655,0.000000,Valid,0.074669,0.146021,0.986460,...,6,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001


In [6]:
df

,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,trial_number,model_visibility_state,object_position_x,object_position_y,object_position_z,object_rotation_x,object_rotation_y,object_rotation_z,object_rotation_w,participant_id
0,1000001481958514100,1778060264384,592.3813,2.000000,232023,0.000000,Valid,-0.097741,0.230809,0.968077,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
1,1000001481963514900,1778060264384,592.3813,2.000000,232024,0.000000,Valid,-0.097410,0.230441,0.968198,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
2,1000001481968515700,1778060264384,592.3813,2.000000,232025,0.000000,Valid,-0.097012,0.229992,0.968345,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
3,1000001481973516500,1778060264384,592.3813,2.000000,232026,0.000000,Valid,-0.096599,0.229737,0.968447,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
4,1000001481978516800,1778060264384,592.3813,2.000000,232027,0.000000,Valid,-0.096269,0.230169,0.968377,...,0,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553473,1000005111509028100,1778063893701,3746.4670,1.157945,957652,0.027887,Valid,0.076445,0.145445,0.986409,...,6,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
553474,1000005111514028600,1778063893701,3746.4670,1.148934,957653,0.000000,Valid,0.075853,0.145398,0.986461,...,6,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
553475,1000005111519029800,1778063893711,3746.4780,1.150903,957654,0.000000,Valid,0.075171,0.145675,0.986472,...,6,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001
553476,1000005111524031000,1778063893711,3746.4780,1.141437,957655,0.000000,Valid,0.074669,0.146021,0.986460,...,6,0,3.25,1.52,-0.11,0.0,-0.707107,0.0,0.707107,001


In [9]:
df["trial_number"].value_counts()

trial_number
2    106543
4    102036
1     82245
3     71251
6     67567
5     66770
0     57066
Name: count, dtype: int64

In [10]:
df["condition_number"].value_counts()

condition_number
2    211218
3    184443
1    157817
Name: count, dtype: int64

In [11]:
df["participant_id"].value_counts()

participant_id
001    553478
Name: count, dtype: int64